# 서울시 도로구간 시각화 및 25m 포인트 생성

`(도로명주소)도로구간_서울/TL_SPRD_MANAGE_11_202605.shp`에서 서울시(`SIG_CD`가 `11`로 시작) 도로구간 중 도로 클래스 `3: 로`, `4: 길`만 추출해 단색으로 시각화하고, 도로 라인 위에 25m 간격 포인트를 생성합니다.


In [1]:
from pathlib import Path
import importlib
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

SRC_DIR = Path.cwd() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from silverwalk import accidents, bus_stops, business, config, crosswalks, crosswalk_warnings, features, intersections, population, roads, social_welfare, speed, speed_bumps, traditional_market, traffic_signals

# 노트북 커널이 이전 버전의 모듈을 캐시할 수 있으므로, 실행할 때마다 최신 파일을 다시 불러옵니다.
for module in [config, roads, features, accidents, speed, business, population, social_welfare, traditional_market, bus_stops, speed_bumps, intersections, crosswalks, crosswalk_warnings, traffic_signals]:
    importlib.reload(module)

from silverwalk.accidents import add_accident_risk
from silverwalk.bus_stops import add_bus_stop_counts
from silverwalk.business import add_business_category_counts
from silverwalk.config import (
    BUFFER_50M,
    ACCIDENT_CSV,
    BUSINESS_CATEGORY_COLUMNS,
    BUSINESS_CSV,
    BUFFER_300M,
    BUS_STOP_COLUMN,
    BUS_STOP_XLSX,
    CROSSWALK_COLUMN,
    CROSSWALK_SHP,
    CROSSWALK_WARNING_COLUMN,
    CROSSWALK_WARNING_SHP,
    ELDERLY_POPULATION_COLUMN,
    INTERSECTION_COLUMN,
    INTERSECTION_SHP,
    NODELINK_SHP,
    PEDESTRIAN_SIGNAL_CSV,
    OUTPUT_JOIN_CSV,
    POINT_INTERVAL_M,
    POPULATION_GRID_DIR,
    SOCIAL_WELFARE_COLUMN,
    SOCIAL_WELFARE_GEOCODED_CSV,
    SPEED_BUMP_COLUMN,
    SPEED_BUMP_SHP,
    TARGET_REGION_NAME,
    TRAFFIC_SIGNAL_COLUMN,
    TRADITIONAL_MARKET_COLUMN,
    TRADITIONAL_MARKET_SHP,
)
from silverwalk.crosswalks import add_crosswalk_counts
from silverwalk.crosswalk_warnings import add_crosswalk_warning_presence
from silverwalk.features import make_base_point_df
from silverwalk.intersections import add_intersection_counts
from silverwalk.population import add_elderly_population_300m
from silverwalk.social_welfare import add_social_welfare_facility_counts
from silverwalk.traditional_market import add_traditional_market_presence
from silverwalk.traffic_signals import add_traffic_signal_presence
from silverwalk.roads import (
    create_point_buffers,
    create_road_points,
    filter_target_roads,
    load_road_segments,
    make_road_summary,
)
from silverwalk.speed import add_speed_limit
from silverwalk.speed_bumps import add_speed_bump_counts

# 한글 폰트가 설치된 환경에서는 한글 제목/범례가 정상 표시됩니다.
plt.rcParams["font.family"] = ["NanumGothic", "Malgun Gothic", "AppleGothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


In [2]:
USE_EXISTING_FINAL_CSV = OUTPUT_JOIN_CSV.exists()
existing_final_df = None
roads = None
target_all_roads = None
target_roads = None
point_buffers_50m_gdf = None

if USE_EXISTING_FINAL_CSV:
    existing_final_df = pd.read_csv(OUTPUT_JOIN_CSV, encoding="utf-8-sig")
    required_point_cols = ["POINT_ID", "위도", "경도"]
    missing_point_cols = [col for col in required_point_cols if col not in existing_final_df.columns]
    if missing_point_cols:
        raise ValueError(f"기존 최종 CSV에 포인트 복원용 컬럼이 없습니다: {missing_point_cols}")

    points_25m_gdf = gpd.GeoDataFrame(
        existing_final_df[["POINT_ID"]].copy(),
        geometry=gpd.points_from_xy(existing_final_df["경도"], existing_final_df["위도"]),
        crs="EPSG:4326",
    ).to_crs(epsg=5179)

    print(f"기존 최종 CSV를 읽었습니다: {OUTPUT_JOIN_CSV}")
    print(f"기존 최종 CSV 행 수: {len(existing_final_df):,}")
    print(f"기존 최종 CSV 컬럼 수: {len(existing_final_df.columns):,}")
    print("도로 shapefile 로드와 25m 포인트 재생성을 건너뜁니다.")
    display(existing_final_df.head())
else:
    roads, ROAD_SHP = load_road_segments()

    print(f"도로구간 파일: {ROAD_SHP}")
    print(f"전체 도로구간 수: {len(roads):,}")
    print(f"원본 좌표계: {roads.crs}")
    display(roads.head())


기존 최종 CSV를 읽었습니다: Data/seoul_road_points.csv
기존 최종 CSV 행 수: 450,663
기존 최종 CSV 컬럼 수: 97
도로 shapefile 로드와 25m 포인트 재생성을 건너뜁니다.


,POINT_ID,위도,경도,제한속도,위험도,고령인구수,사회복지시설개수,전통시장여부,버스정류장개수,과속방지턱개수,...,음식_구내식당·뷔페,음식_기타 간이,음식_기타 외국,음식_동남아시아,음식_비알코올,음식_서양식,음식_일식,음식_주점,음식_중식,음식_한식
0,0,37.591080,126.992864,10,0.0,913,0,0,11,0,...,2,7,0,1,8,1,2,0,1,12
1,1,37.590982,126.992616,10,0.0,913,0,0,11,0,...,2,6,0,1,7,1,2,0,0,10
2,2,37.590820,126.992421,10,0.0,913,0,0,12,0,...,2,6,0,1,6,1,2,0,0,10
3,3,37.590660,126.992225,10,0.0,913,0,0,11,0,...,1,6,0,1,6,1,2,0,0,10
4,4,37.590492,126.992037,10,0.0,913,0,0,13,0,...,1,6,0,1,6,1,2,0,0,10


In [3]:
if USE_EXISTING_FINAL_CSV:
    print("기존 최종 CSV를 사용하므로 도로 필터링을 건너뜁니다.")
else:
    target_all_roads, target_roads = filter_target_roads(roads)
    summary = make_road_summary(target_all_roads, target_roads)

    display(summary)
    display(target_roads["ROA_CLS_SE"].value_counts().sort_index().rename("도로 클래스별 구간 수"))
    display(target_roads[["SIG_CD", "RN", "RN_CD", "ROAD_LT", "ROAD_BT", "ROA_CLS_SE", "geometry"]].head())


기존 최종 CSV를 사용하므로 도로 필터링을 건너뜁니다.


In [4]:
if USE_EXISTING_FINAL_CSV:
    print("기존 최종 CSV를 사용하므로 도로 시각화를 건너뜁니다.")
else:
    fig, ax = plt.subplots(figsize=(10, 10))

    target_roads.plot(
        ax=ax,
        linewidth=0.45,
        color="#2563eb",
        alpha=0.85,
    )

    ax.set_title(f"{TARGET_REGION_NAME} Road Network - Classes 3 and 4", fontsize=16, pad=12)
    ax.set_axis_off()
    ax.set_aspect("equal")

    plt.tight_layout()
    plt.show()


기존 최종 CSV를 사용하므로 도로 시각화를 건너뜁니다.


## 서울시 도로 25m 간격 포인트 생성

현재 도로 데이터 좌표계는 EPSG:5179이므로 거리 단위는 미터입니다. 아래 코드는 도로 클래스 `3: 로`, `4: 길` 라인 위에 25m 간격 포인트를 생성합니다.


In [5]:
if USE_EXISTING_FINAL_CSV:
    print(f"기존 최종 CSV의 위도/경도로 복원한 포인트 수: {len(points_25m_gdf):,}")
    display(points_25m_gdf.head())
else:
    points_25m_gdf = create_road_points(target_roads, distance=POINT_INTERVAL_M)

    print(f"생성된 25m 간격 포인트 수: {len(points_25m_gdf):,}")
    display(points_25m_gdf.head())


기존 최종 CSV의 위도/경도로 복원한 포인트 수: 450,663


,POINT_ID,geometry
0,0,POINT (955228.522 1954751.951)
1,1,POINT (955206.65 1954741.134)
2,2,POINT (955189.328 1954723.282)
3,3,POINT (955171.912 1954705.586)
4,4,POINT (955155.168 1954687.123)


In [6]:
if USE_EXISTING_FINAL_CSV:
    print("기존 최종 CSV를 사용하므로 도로 포인트 시각화를 건너뜁니다.")
else:
    fig, ax = plt.subplots(figsize=(10, 10))

    target_roads.plot(
        ax=ax,
        linewidth=0.35,
        color="#94a3b8",
        alpha=0.7,
    )

    points_25m_gdf.plot(
        ax=ax,
        markersize=0.25,
        color="#dc2626",
        alpha=0.75,
    )

    ax.set_title(f"{TARGET_REGION_NAME} Road Points at 25m Intervals - Classes 3 and 4", fontsize=16, pad=12)
    ax.set_axis_off()
    ax.set_aspect("equal")

    plt.tight_layout()
    plt.show()


기존 최종 CSV를 사용하므로 도로 포인트 시각화를 건너뜁니다.


## 25m 포인트별 50m 버퍼 생성

`points_25m_gdf`의 각 포인트를 중심으로 반경 50m 버퍼를 생성합니다. EPSG:5179 좌표계는 미터 단위이므로 `buffer(50)`을 사용합니다.

In [7]:
if USE_EXISTING_FINAL_CSV:
    print("기존 최종 CSV를 사용하므로 50m 버퍼는 필요한 경우 최종 결합 셀에서 생성합니다.")
else:
    point_buffers_50m_gdf = create_point_buffers(points_25m_gdf, buffer_distance_m=BUFFER_50M)

    print(f"생성된 50m 버퍼 수: {len(point_buffers_50m_gdf):,}")
    display(point_buffers_50m_gdf.head())


기존 최종 CSV를 사용하므로 50m 버퍼는 필요한 경우 최종 결합 셀에서 생성합니다.


In [8]:
if USE_EXISTING_FINAL_CSV:
    print("기존 최종 CSV를 사용하므로 50m 버퍼 시각화를 건너뜁니다.")
else:
    fig, ax = plt.subplots(figsize=(10, 10))

    point_buffers_50m_gdf.plot(
        ax=ax,
        color="#f97316",
        alpha=0.12,
        edgecolor="none",
    )

    target_roads.plot(
        ax=ax,
        linewidth=0.25,
        color="#334155",
        alpha=0.75,
    )

    points_25m_gdf.plot(
        ax=ax,
        markersize=0.15,
        color="#dc2626",
        alpha=0.6,
    )

    ax.set_title(f"50m Buffers Around {TARGET_REGION_NAME} 25m Road Points - Classes 3 and 4", fontsize=16, pad=12)
    ax.set_axis_off()
    ax.set_aspect("equal")

    plt.tight_layout()
    plt.show()


기존 최종 CSV를 사용하므로 50m 버퍼 시각화를 건너뜁니다.


## Feature 추가 함수

Feature 추가 함수는 `src/silverwalk/` 모듈로 분리했습니다. 이 노트북에서는 결합 순서만 실행합니다.


In [9]:
print(f"상가 업종 컬럼 수: {len(BUSINESS_CATEGORY_COLUMNS):,}")
print(f"사고 데이터: {ACCIDENT_CSV}")
print(f"ITS 표준노드링크: {NODELINK_SHP}")
print(f"상가 정보: {BUSINESS_CSV}")
print(f"고령인구 격자 데이터: {POPULATION_GRID_DIR}")
print(f"노인의료복지시설 지오코딩 CSV: {SOCIAL_WELFARE_GEOCODED_CSV}")
print(f"전통시장 데이터: {TRADITIONAL_MARKET_SHP}")
print(f"버스정류장 데이터: {BUS_STOP_XLSX}")
print(f"과속방지턱 데이터: {SPEED_BUMP_SHP}")
print(f"교차로 데이터: {INTERSECTION_SHP}")
print(f"횡단보도 데이터: {CROSSWALK_SHP}")
print(f"횡단보도예고표시 데이터: {CROSSWALK_WARNING_SHP}")
print(f"보행자신호등 데이터: {PEDESTRIAN_SIGNAL_CSV}")


상가 업종 컬럼 수: 85
사고 데이터: Data/seoul_old_pedestrian_individual_accidents_2020_2025.csv
ITS 표준노드링크: Data/ITS_node_link/MOCT_LINK.shp
상가 정보: Data/소상공인시장진흥공단_상가(상권)정보_서울_202603.csv
고령인구 격자 데이터: Data/국토통계_고령인구수
노인의료복지시설 지오코딩 CSV: Data/서울시_노인의료복지시설현황_geocoded.csv
전통시장 데이터: Data/전통시장여부/서울시 상권분석서비스(영역-상권)_전통시장.shp
버스정류장 데이터: Data/버스정류장개수/서울시버스정류소위치정보(20241002).xlsx
과속방지턱 데이터: Data/과속방지턱개수/A067_A.shp
교차로 데이터: Data/교차로개수/A008_P_20250814/A008_P/A008_P.shp
횡단보도 데이터: Data/횡단보도개수/횡단보도 위치 및 부착대 정보/A004_A_횡단보도/A004_A.shp
횡단보도예고표시 데이터: Data/횡단보도예고표시/A068_P(20250417)/A068_P.shp


## 최종 CSV 생성 및 저장

최종 CSV가 이미 있으면 해당 CSV를 읽고, 없는 feature 컬럼만 추가 계산한 뒤 다시 저장합니다.


In [10]:
BASE_COLUMNS = ["POINT_ID", "위도", "경도"]
FEATURE_COLUMNS = ["제한속도", "위험도", ELDERLY_POPULATION_COLUMN, SOCIAL_WELFARE_COLUMN, TRADITIONAL_MARKET_COLUMN, BUS_STOP_COLUMN, SPEED_BUMP_COLUMN, INTERSECTION_COLUMN, CROSSWALK_COLUMN, CROSSWALK_WARNING_COLUMN, TRAFFIC_SIGNAL_COLUMN] + BUSINESS_CATEGORY_COLUMNS
FINAL_COLUMNS = BASE_COLUMNS + FEATURE_COLUMNS

if OUTPUT_JOIN_CSV.exists():
    final_join_df = pd.read_csv(OUTPUT_JOIN_CSV, encoding="utf-8-sig")
    print(f"기존 최종 CSV에서 시작합니다: {OUTPUT_JOIN_CSV}")
else:
    final_join_df = make_base_point_df(points_25m_gdf)
    print("기존 최종 CSV가 없어 25m 포인트 기본 데이터부터 생성합니다.")

# 기본 컬럼 확인
missing_base_cols = [col for col in BASE_COLUMNS if col not in final_join_df.columns]
if missing_base_cols:
    base_point_df = make_base_point_df(points_25m_gdf)
    final_join_df = final_join_df.drop(columns=[col for col in BASE_COLUMNS if col in final_join_df.columns], errors="ignore")
    final_join_df = final_join_df.merge(base_point_df, on="POINT_ID", how="left")
    print(f"기본 좌표 컬럼 추가: {missing_base_cols}")
else:
    print("기본 좌표 컬럼이 이미 있어 건너뜁니다.")

# 위험도 
if "위험도" not in final_join_df.columns:
    if point_buffers_50m_gdf is None:
        point_buffers_50m_gdf = create_point_buffers(points_25m_gdf, buffer_distance_m=BUFFER_50M)
    final_join_df = add_accident_risk(
        final_df=final_join_df,
        points_gdf=points_25m_gdf,
        point_buffers_gdf=point_buffers_50m_gdf,
        accident_csv=ACCIDENT_CSV,
    )
else:
    print("위험도 컬럼이 이미 있어 건너뜁니다.")

# 제한속도
if "제한속도" not in final_join_df.columns:
    final_join_df = add_speed_limit(
        final_df=final_join_df,
        points_gdf=points_25m_gdf,
        nodelink_shp=NODELINK_SHP,
    )
else:
    print("제한속도 컬럼이 이미 있어 건너뜁니다.")

# 상가 업종별 개수
missing_business_cols = [col for col in BUSINESS_CATEGORY_COLUMNS if col not in final_join_df.columns]
if missing_business_cols:
    final_join_df = final_join_df.drop(
        columns=[col for col in BUSINESS_CATEGORY_COLUMNS if col in final_join_df.columns],
        errors="ignore",
    )
    final_join_df = add_business_category_counts(
        final_df=final_join_df,
        points_gdf=points_25m_gdf,
        business_csv=BUSINESS_CSV,
        radius_m=BUFFER_300M,
    )
else:
    print("상가 업종 컬럼이 모두 있어 건너뜁니다.")

# 고령인구 컬럼
if ELDERLY_POPULATION_COLUMN not in final_join_df.columns:
    final_join_df = add_elderly_population_300m(
        final_df=final_join_df,
        points_gdf=points_25m_gdf,
        radius_m=BUFFER_300M,
    )
else:
    print(f"{ELDERLY_POPULATION_COLUMN} 컬럼이 이미 있어 건너뜁니다.")

# 사회복지시설 컬럼
if SOCIAL_WELFARE_COLUMN not in final_join_df.columns:
    final_join_df = add_social_welfare_facility_counts(
        final_df=final_join_df,
        points_gdf=points_25m_gdf,
        geocoded_csv=SOCIAL_WELFARE_GEOCODED_CSV,
        radius_m=BUFFER_300M,
    )
else:
    print(f"{SOCIAL_WELFARE_COLUMN} 컬럼이 이미 있어 건너뜁니다.")

# 전통시장 컬럼
if TRADITIONAL_MARKET_COLUMN not in final_join_df.columns:
    final_join_df = add_traditional_market_presence(
        final_df=final_join_df,
        points_gdf=points_25m_gdf,
        market_shp=TRADITIONAL_MARKET_SHP,
        radius_m=BUFFER_50M,
    )
else:
    print(f"{TRADITIONAL_MARKET_COLUMN} 컬럼이 이미 있어 건너뜁니다.")

# 버스정류장 컬럼
if BUS_STOP_COLUMN not in final_join_df.columns:
    final_join_df = add_bus_stop_counts(
        final_df=final_join_df,
        points_gdf=points_25m_gdf,
        bus_stop_xlsx=BUS_STOP_XLSX,
        radius_m=BUFFER_300M,
    )
else:
    print(f"{BUS_STOP_COLUMN} 컬럼이 이미 있어 건너뜁니다.")

# 과속방지턱 컬럼
if SPEED_BUMP_COLUMN not in final_join_df.columns:
    final_join_df = add_speed_bump_counts(
        final_df=final_join_df,
        points_gdf=points_25m_gdf,
        speed_bump_shp=SPEED_BUMP_SHP,
        radius_m=BUFFER_50M,
    )
else:
    print(f"{SPEED_BUMP_COLUMN} 컬럼이 이미 있어 건너뜁니다.")

# 교차로 컬럼
if INTERSECTION_COLUMN not in final_join_df.columns:
    final_join_df = add_intersection_counts(
        final_df=final_join_df,
        points_gdf=points_25m_gdf,
        intersection_shp=INTERSECTION_SHP,
        radius_m=BUFFER_50M,
    )
else:
    print(f"{INTERSECTION_COLUMN} 컬럼이 이미 있어 건너뜁니다.")

# 횡단보도 컬럼
if CROSSWALK_COLUMN not in final_join_df.columns:
    final_join_df = add_crosswalk_counts(
        final_df=final_join_df,
        points_gdf=points_25m_gdf,
        crosswalk_shp=CROSSWALK_SHP,
        radius_m=BUFFER_50M,
    )
else:
    print(f"{CROSSWALK_COLUMN} 컬럼이 이미 있어 건너뜁니다.")

# 횡단보도예고표시 컬럼
if CROSSWALK_WARNING_COLUMN not in final_join_df.columns:
    final_join_df = add_crosswalk_warning_presence(
        final_df=final_join_df,
        points_gdf=points_25m_gdf,
        crosswalk_warning_shp=CROSSWALK_WARNING_SHP,
        radius_m=BUFFER_50M,
    )
else:
    print(f"{CROSSWALK_WARNING_COLUMN} 컬럼이 이미 있어 건너뜁니다.")

# 보행자신호등 설치여부 컬럼
if TRAFFIC_SIGNAL_COLUMN not in final_join_df.columns:
    final_join_df = add_traffic_signal_presence(
        final_df=final_join_df,
        points_gdf=points_25m_gdf,
        pedestrian_signal_csv=PEDESTRIAN_SIGNAL_CSV,
        radius_m=BUFFER_50M,
    )
else:
    print(f"{TRAFFIC_SIGNAL_COLUMN} 컬럼이 이미 있어 건너뜁니다.")

# 최종 컬럼 확인
missing_final_cols = [col for col in FINAL_COLUMNS if col not in final_join_df.columns]
if missing_final_cols:
    raise ValueError(f"최종 CSV 저장 전 누락된 컬럼이 있습니다: {missing_final_cols}")

# 최종 CSV 저장
final_join_df = final_join_df[FINAL_COLUMNS]

# 저장 경로의 상위 디렉토리가 없는 경우 생성한 후 저장합니다.
OUTPUT_JOIN_CSV.parent.mkdir(parents=True, exist_ok=True)
final_join_df.to_csv(OUTPUT_JOIN_CSV, index=False, encoding="utf-8-sig")


# 최종 데이터 정보 출력
print(f"저장된 최종 데이터 행 수: {len(final_join_df):,}")
print(f"저장된 최종 데이터 컬럼 수: {len(final_join_df.columns):,}")
print(f"제한속도 결측 수: {final_join_df['제한속도'].isna().sum():,}")
print(f"상가 업종 컬럼 수: {len(BUSINESS_CATEGORY_COLUMNS):,}")
print(f"고령인구 컬럼: {ELDERLY_POPULATION_COLUMN}")
print(f"사회복지시설 컬럼: {SOCIAL_WELFARE_COLUMN}")
print(f"전통시장 컬럼: {TRADITIONAL_MARKET_COLUMN}")
print(f"버스정류장 컬럼: {BUS_STOP_COLUMN}")
print(f"과속방지턱 컬럼: {SPEED_BUMP_COLUMN}")
print(f"교차로 컬럼: {INTERSECTION_COLUMN}")
print(f"횡단보도 컬럼: {CROSSWALK_COLUMN}")
print(f"횡단보도예고표시 컬럼: {CROSSWALK_WARNING_COLUMN}")
print(f"보행자신호등 컬럼: {TRAFFIC_SIGNAL_COLUMN}")
print(f"CSV 저장: {OUTPUT_JOIN_CSV}")
display(final_join_df.head())


기존 최종 CSV에서 시작합니다: Data/seoul_road_points.csv
기본 좌표 컬럼이 이미 있어 건너뜁니다.
위험도 컬럼이 이미 있어 건너뜁니다.
제한속도 컬럼이 이미 있어 건너뜁니다.
상가 업종 컬럼이 모두 있어 건너뜁니다.
고령인구수 컬럼이 이미 있어 건너뜁니다.
사회복지시설개수 컬럼이 이미 있어 건너뜁니다.
전통시장여부 컬럼이 이미 있어 건너뜁니다.
버스정류장개수 컬럼이 이미 있어 건너뜁니다.
과속방지턱개수 컬럼이 이미 있어 건너뜁니다.
교차로개수 컬럼이 이미 있어 건너뜁니다.
횡단보도개수 컬럼이 이미 있어 건너뜁니다.
서울시 횡단보도예고표시 point 수: 35,476
반경 50m 횡단보도예고표시-포인트 매칭 수: 303,813
횡단보도예고표시 포함 POINT 수: 117,066
저장된 최종 데이터 행 수: 450,663
저장된 최종 데이터 컬럼 수: 98
제한속도 결측 수: 0
상가 업종 컬럼 수: 85
고령인구 컬럼: 고령인구수
사회복지시설 컬럼: 사회복지시설개수
전통시장 컬럼: 전통시장여부
버스정류장 컬럼: 버스정류장개수
과속방지턱 컬럼: 과속방지턱개수
교차로 컬럼: 교차로개수
횡단보도 컬럼: 횡단보도개수
횡단보도예고표시 컬럼: 횡단보도예고표시여부
CSV 저장: Data/seoul_road_points.csv


,POINT_ID,위도,경도,제한속도,위험도,고령인구수,사회복지시설개수,전통시장여부,버스정류장개수,과속방지턱개수,...,음식_구내식당·뷔페,음식_기타 간이,음식_기타 외국,음식_동남아시아,음식_비알코올,음식_서양식,음식_일식,음식_주점,음식_중식,음식_한식
0,0,37.591080,126.992864,10,0.0,913,0,0,11,0,...,2,7,0,1,8,1,2,0,1,12
1,1,37.590982,126.992616,10,0.0,913,0,0,11,0,...,2,6,0,1,7,1,2,0,0,10
2,2,37.590820,126.992421,10,0.0,913,0,0,12,0,...,2,6,0,1,6,1,2,0,0,10
3,3,37.590660,126.992225,10,0.0,913,0,0,11,0,...,1,6,0,1,6,1,2,0,0,10
4,4,37.590492,126.992037,10,0.0,913,0,0,13,0,...,1,6,0,1,6,1,2,0,0,10
